# Student Performance Prediction using Linear Regression
**Minor Project 1 | Supervised Machine Learning**

**Dataset:** Students Performance in Exams (Kaggle)  
**Dataset Link:** https://www.kaggle.com/datasets/spscientist/students-performance-in-exams
**Objective:** Predict a student's math score based on demographic and academic factors.

## 1. Import Libraries

In [ ]:
# pip install pandas numpy matplotlib seaborn scikit-learn joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')
print('Libraries imported successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/StudentsPerformance.csv')
print('Dataset Shape:', df.shape)
df.head()

## 3. Data Preprocessing

In [ ]:
# Dataset Info
df.info()

In [ ]:
# Descriptive Statistics
df.describe()

In [ ]:
# Check for missing values
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
# Check for duplicates
print('Duplicate Rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)
print('Shape after removing duplicates:', df.shape)

In [ ]:
# Feature Encoding - Label Encoding for categorical columns
df_enc = df.copy()
le = LabelEncoder()
categorical_cols = df_enc.select_dtypes(include='object').columns.tolist()
print('Categorical Columns:', categorical_cols)
for col in categorical_cols:
    df_enc[col] = le.fit_transform(df_enc[col])
print('\nEncoded Dataset:')
df_enc.head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of Math Score
plt.figure(figsize=(8, 5))
sns.histplot(df['math score'], kde=True, color='steelblue')
plt.title('Distribution of Math Scores', fontsize=14, fontweight='bold')
plt.xlabel('Math Score')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('../results/eda_math_score_distribution.png', dpi=150)
plt.show()

In [ ]:
# Math Score by Gender
plt.figure(figsize=(7, 5))
sns.boxplot(x='gender', y='math score', data=df, hue='gender', palette='Set2', legend=False)
plt.title('Math Score by Gender', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/eda_math_score_by_gender.png', dpi=150)
plt.show()

In [ ]:
# Math Score by Test Preparation Course
plt.figure(figsize=(7, 5))
sns.boxplot(x='test preparation course', y='math score', data=df, hue='test preparation course', palette='Set1', legend=False)
plt.title('Math Score by Test Preparation Course', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/eda_math_score_by_test_prep.png', dpi=150)
plt.show()

In [ ]:
# Math Score by Parental Level of Education
plt.figure(figsize=(10, 5))
order = df.groupby('parental level of education')['math score'].median().sort_values(ascending=False).index
sns.boxplot(x='parental level of education', y='math score', data=df, order=order, hue='parental level of education', palette='coolwarm', legend=False)
plt.title('Math Score by Parental Level of Education', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../results/eda_math_score_by_parental_edu.png', dpi=150)
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(9, 7))
sns.heatmap(df_enc.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/eda_correlation_heatmap.png', dpi=150)
plt.show()

## 5. Model Development — Linear Regression

In [ ]:
# Define Features and Target
X = df_enc.drop('math score', axis=1)
y = df_enc['math score']
print('Features:', X.columns.tolist())
print('Target: math score')

In [ ]:
# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Training Samples:', X_train.shape[0])
print('Testing  Samples:', X_test.shape[0])

In [ ]:
# Train the Model
model = LinearRegression()
model.fit(X_train, y_train)
print('Model Trained Successfully!')
print('\nModel Coefficients:')
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
print(coef_df.sort_values('Coefficient', ascending=False).to_string(index=False))
print(f'\nIntercept: {model.intercept_:.4f}')

## 6. Model Evaluation

In [ ]:
# Predictions & Metrics
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('=' * 35)
print('       Evaluation Metrics')
print('=' * 35)
print(f'  MAE      : {mae:.4f}')
print(f'  MSE      : {mse:.4f}')
print(f'  RMSE     : {rmse:.4f}')
print(f'  R² Score : {r2:.4f}')
print('=' * 35)

# Save metrics
metrics = pd.DataFrame({'Metric': ['MAE', 'MSE', 'RMSE', 'R² Score'],
                        'Value': [round(mae,4), round(mse,4), round(rmse,4), round(r2,4)]})
metrics.to_csv('../results/evaluation_metrics.csv', index=False)

In [ ]:
# Actual vs Predicted Plot
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6, edgecolors='k', linewidths=0.3, color='steelblue', label='Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect Fit')
plt.xlabel('Actual Math Score')
plt.ylabel('Predicted Math Score')
plt.title('Actual vs Predicted Math Scores', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../results/evaluation_actual_vs_predicted.png', dpi=150)
plt.show()

In [ ]:
# Residuals Plot
residuals = y_test - y_pred
plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals, alpha=0.5, color='coral', edgecolors='k', linewidths=0.3)
plt.axhline(0, color='black', linewidth=1.5, linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residuals Plot', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/evaluation_residuals.png', dpi=150)
plt.show()

In [ ]:
# Sample: Actual vs Predicted Table
result = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred.round(2)})
result.reset_index(drop=True, inplace=True)
print(result.head(20))

In [ ]:
# Save trained model
joblib.dump(model, '../model/linear_regression_model.pkl')
print('Model saved to ../model/linear_regression_model.pkl')

## 7. Conclusion

- A **Linear Regression** model was trained to predict students' math scores from demographic and academic features.
- The model achieved an **R² Score of 0.8838**, meaning it explains approximately **88.4%** of the variance in math scores.
- **Reading score** and **writing score** are the strongest predictors of math performance.
- Students who completed test preparation courses scored higher on average.
- Higher parental education levels correlate with better student performance.
- The low RMSE (≈5.32) indicates predictions are generally within 5 points of actual scores, which is acceptable for this domain.